### Structured Output in LangChain

Structured output allows agents to return data in a specific, predictable format. Instead of parsing natural language responses, you get structured data in the form of JSON objects, Pydantic models, or dataclasses that your application can use directly.

Prerequisite : Pydantic

---

### Core Concepts of Structured Output

* Schema Enforcement: Forces the LLM to respond strictly matching a defined schema.
* Type Safety: Returned objects are validated Python objects with type hints.
* Methods: Works via with_structured_output() using Pydantic, TypedDict, or JSON Schema.
* Production Readiness: Eliminates fragile regex string parsing when building downstream pipelines.

### Structured Output Method Matrix

| Schema Type | Description | Return Type |
| :--- | :--- | :--- |
| Pydantic BaseModel | Full validation with field descriptions, default values, and type checks. | Pydantic model instance |
| TypedDict | Lightweight type annotations for returning plain Python dictionaries. | Python dict |
| JSON Schema | Raw JSON schema dictionary definition. | Python dict |

### 1. Environment and Model Setup

In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()

model = init_chat_model("gemini-3.6-flash", model_provider="google_genai")

### 2. Structured Output with Pydantic BaseModel

Pydantic is the recommended way to define structured schemas in LangChain because Field descriptions guide the LLM on what each attribute represents.

In [2]:
from pydantic import BaseModel, Field
from typing import List, Optional

class MovieReview(BaseModel):
    title: str = Field(description="Title of the movie")
    director: str = Field(description="Name of the director")
    rating: float = Field(description="Rating score from 1.0 to 10.0")
    summary: str = Field(description="Brief 2-sentence summary of the movie")
    pros: List[str] = Field(description="List of top 3 strengths or highlights")
    cons: List[str] = Field(description="List of weaknesses or critiques")
    recommend: bool = Field(description="True if recommended, False otherwise")

# Bind schema to model
structured_llm = model.with_structured_output(MovieReview)

# Invoke with raw text prompt
review_text = """
Inception (2010), directed by Christopher Nolan, is an absolute masterpiece of modern sci-fi cinema.
The visual effects and Hans Zimmer score are mind-blowing, and the multi-layered dream mechanics are brilliant.
However, some exposition-heavy dialogue in the middle act can feel a bit overwhelming on first watch.
Overall, I give it a 9.5 out of 10 and highly recommend it to everyone.
"""

result = structured_llm.invoke(review_text)

print(f"Type of result: {type(result).__name__}")
print(f"Title: {result.title}")
print(f"Director: {result.director}")
print(f"Rating: {result.rating}/10")
print(f"Pros: {result.pros}")
print(f"Cons: {result.cons}")
print(f"Recommend: {result.recommend}")

Type of result: MovieReview
Title: Inception
Director: Christopher Nolan
Rating: 9.5/10
Pros: ['Mind-blowing visual effects', 'Exceptional Hans Zimmer score', 'Brilliant multi-layered dream mechanics']
Cons: ['Exposition-heavy dialogue in the middle act can feel overwhelming']
Recommend: True


### 3. Structured Output with TypedDict

If you prefer returning plain Python dictionaries instead of Pydantic instances, you can pass a TypedDict schema to with_structured_output():

In [3]:
from typing import TypedDict, List

class PersonInfo(TypedDict):
    name: str
    age: int
    skills: List[str]
    current_role: str

structured_dict_llm = model.with_structured_output(PersonInfo)

prompt = "Alice is a 29 year old Senior Data Scientist who specializes in Python, PyTorch, and SQL."

data = structured_dict_llm.invoke(prompt)

print(f"Type of output: {type(data).__name__}")
print(f"Data: {data}")
print(f"Name: {data.get('name')}")
print(f"Skills: {data.get('skills')}")

Type of output: dict
Data: {'name': 'Alice', 'age': 29, 'skills': ['Python', 'PyTorch', 'SQL'], 'current_role': 'Senior Data Scientist'}
Name: Alice
Skills: ['Python', 'PyTorch', 'SQL']


### 4. Accessing Raw Message Metadata with include_raw

By default, with_structured_output() returns only the parsed object. Setting include_raw=True returns a dictionary containing the raw AIMessage alongside the parsed result:

In [4]:
class ArticleSummary(BaseModel):
    headline: str = Field(description="Catchy main headline")
    key_takeaways: List[str] = Field(description="Top 3 key takeaways")

raw_llm = model.with_structured_output(ArticleSummary, include_raw=True)

article = "LangChain simplified LLM application development by standardizing interfaces across providers and enabling structured output parsing."

output = raw_llm.invoke(article)

print("Keys in output dictionary:", list(output.keys()))
print("Parsed Headline:", output["parsed"].headline)
print("Raw AIMessage ID:", output["raw"].id)

Keys in output dictionary: ['raw', 'parsed', 'parsing_error']
Parsed Headline: How LangChain Simplifies LLM Application Development
Raw AIMessage ID: lc_run--019f97f0-ecf8-7591-8870-fc11899439e7-0
